In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, f1_score

In [ ]:
# Load the three datasets
print("Loading datasets...")
try:
    train_df = pd.read_csv(MACHINE_A_TRAIN_SET_FILE)
    val_df   = pd.read_csv(MACHINE_A_VAL_SET_FILE)
    test_df  = pd.read_csv(MACHINE_A_TEST_SET_FILE)
    
    X_train = train_df[MACHINE_A_FEATURES]
    y_train = train_df['label']

    X_val   = val_df[MACHINE_A_FEATURES]
    y_val   = val_df['label']

    X_test  = test_df[MACHINE_A_FEATURES]
    y_test  = test_df['label']

        
    print(f"Loaded successfully.")
    print(f"Train: {train_df.shape}")
    print(f"Val:   {val_df.shape}")
    print(f"Test:  {test_df.shape}")

except FileNotFoundError as e:
    print(f"ERROR: Could not find split files. Did you run Notebook 22? \n{e}")

In [ ]:
# Training Loop
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score

# Define the Pipeline
# We use 'balanced' class weights to handle the 2:1 imbalance
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('log_reg', LogisticRegression(
        class_weight='balanced',
        solver='lbfgs',
        max_iter=1000,
        random_state=42
    ))
])

# Define Hyperparameters to test
# C = Inverse Regularization Strength
# Small C (0.001) = Strong Regularization (Prevents Overfitting)
# Large C (100) = Weak Regularization (Trusts data more)
C_candidates = [0.001, 0.01, 0.1, 1, 10, 100]

best_f1 = 0
best_model = None
history = []

print(f"{'C Value':<10} | {'Val F1 Score':<15} | {'Val Accuracy':<15}")
print("-" * 45)

# Training Loop
for c in C_candidates:
    # Set hyperparameter
    pipeline.set_params(log_reg__C=c)

    # Train
    pipeline.fit(X_train, y_train)

    # Validate
    val_preds = pipeline.predict(X_val)
    f1 = f1_score(y_val, val_preds)
    acc = np.mean(val_preds == y_val)

    history.append({'C': c, 'F1': f1})
    print(f"{c:<10} | {f1:.5f}         | {acc:.5f}")

    # Track Best Model
    if f1 > best_f1:
        best_f1 = f1
        # Save a deep copy of the best pipeline
        best_model = joblib.load(joblib.dump(pipeline, 'temp_model.pkl')[0])

print("-" * 45)
print(f"Winner: C={best_model.named_steps['log_reg'].C} with F1={best_f1:.5f}")

In [ ]:
# Final Test & Save
from sklearn.metrics import classification_report, confusion_matrix

# Final Prediction on Test Set
test_preds = best_model.predict(X_test)

print("=== Final Test Set Report ===")
print(classification_report(y_test, test_preds, target_names=['Clean', 'Jamming']))

# Confusion Matrix Plot
cm = confusion_matrix(y_test, test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Clean', 'Pred Jamming'],
            yticklabels=['True Clean', 'True Jamming'])
plt.title('Confusion Matrix (Test Set)')
plt.show()

joblib.dump(best_model, MACHINE_A_MODEL_FILE)

print(f"Model successfully saved to: {MACHINE_A_MODEL_FILE}")